
# 간이 KcBERT: 한국어 BERT 사전학습(MLM) 전체 파이프라인

이 노트북은 한 개의 텍스트 파일(`Pre_training_datasets.txt`)을 이용해 **작은 BERT**를 처음부터 사전학습(MLM) 하는 과정을 담고 있습니다.  
구성:
1. 환경 준비
2. WordPiece 토크나이저 학습
3. 데이터셋 전처리 (MLM용 입력 생성)
4. 작은 BERT 구성 및 학습
5. 저장 및 간이 테스트 (fill-mask)
6. (옵션) NSP/SOP 개념 및 스켈레톤


## 0. 환경 준비

In [3]:
import torch
from pathlib import Path
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizerFast

from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
from transformers import BertConfig, BertForMaskedLM
from transformers import TrainingArguments, Trainer
import math
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForMaskedLM
from datasets import load_dataset
from itertools import chain
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification
from datasets import Dataset
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
import numpy as np
from transformers import TextClassificationPipeline
from sklearn.metrics import confusion_matrix, classification_report

c:\miniconda3\envs\kcbert\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 경로 및 기본 설정

In [ ]:
print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU 이름:", torch.cuda.get_device_name(0))
print("현재 GPU 메모리 사용량 (MB):", torch.cuda.memory_allocated()/1024**2)

CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 4060 Laptop GPU
현재 GPU 메모리 사용량 (MB): 0.0


In [2]:

from pathlib import Path

# 노트북과 같은 폴더에 텍스트 파일을 둔다고 가정합니다.
CORPUS_PATH = Path("Pre_training_dataset.txt")  # 필요시 경로 수정
TOKENIZER_DIR = Path("./kcbert_tokenizer")
MODEL_DIR = Path("./kcbert_mlm")
CHECKPOINT_DIR = MODEL_DIR  # Trainer가 내부에 체크포인트 폴더 생성

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 학습 하이퍼파라미터 (적절히 조정)
VOCAB_SIZE = 32000
LOWER_CASE = False               # 한국어에선 보통 False가 많지만, 소문자 일관화가 유리하면 True 사용
BLOCK_SIZE = 128                # 토큰 시퀀스 길이
MLM_PROB = 0.15                 # 마스크 비율
BATCH_SIZE = 32                 # GPU 메모리에 맞게 조절
EPOCHS = 3                      # 예시용 소량 epoch
LEARNING_RATE = 1.8e-4            # MLM pretrain은 보통 1e-4 ~ 5e-4 범위
WARMUP_RATIO = 0.12
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 250
SAVE_STEPS = 2000
SEED = 7

# 작은 BERT 설정 (대략 base의 절반 이하 크기)
HIDDEN_SIZE = 512
NUM_HIDDEN_LAYERS = 6
NUM_ATTENTION_HEADS = 8
INTERMEDIATE_SIZE = 2048
MAX_POS_EMBED = 512


## 2. 코퍼스 확인

In [3]:

# 파일 존재 여부와 샘플 몇 줄 확인
assert CORPUS_PATH.exists(), f"코퍼스 파일을 찾을 수 없습니다: {CORPUS_PATH.resolve()}"
for i, line in enumerate(open(CORPUS_PATH, "r", encoding="utf-8")):
    print(line.strip())
    if i >= 4:
        break
print("\n... (앞부분 일부만 출력)")


text
국민 사기꾼 전과4범과 더불어 전과자당 국민 혈세 뜯어 먹은 법카 카드깡 주범이나 고소해라 완전 웃기는 인간들이네
지난 4월초에는 동대문전화국이 있는 흥인사거리 및 다산로의 가로변 담장 3개소 250㎡ 면적에 삼색조팝, 수호초 등 2만 6천본을 식재하여 아름다운 꽃벽을 만들어 주민들로부터 좋은 반응을 얻었다.
아직도 검사하는 쫄보 돌대가리들 많네...에효 한심하다.2년 넘게 속았으면 눈치차릴때도 됐는대..저런 사람들이 많을수록 권력쥐고 힘있는 사람들이 해먹기 좋지...
모든 자막은 데이터 변조입니다. 말 한 넘이 나 다른 말 했는데? 하면 다른 말이 되어 버리는데 함부로 감히 자막을 달아놨으니 데이터 변조이죠. 앞으로 녹취록은 무용지물입니다. 거짓말 마음대로 해도 됩니다. 나 그렇게 말 안했어 하고 딴소리 하면 그 말은 이럴수도 저럴수도 있는 말이기 때문에 정확하게 밝힐 수 없는 말이 되어 버립니다. 전국민이 아 라고 들었어도 본인이 어 였다고 하면 그건 논란의 말이 되어 버리고 그 누구도 이거다 라고 명확하게 말하면 가짜뉴스범 데이터 변조범이 되어 버리는 것이죠.

... (앞부분 일부만 출력)


## 3. WordPiece 토크나이저 학습

In [4]:

from tokenizers import BertWordPieceTokenizer

# WordPiece 토크나이저 학습
tokenizer_trainer = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=LOWER_CASE,  # lower_case가 True일 때 보통 strip_accents도 True
    lowercase=LOWER_CASE
)

tokenizer_trainer.train(
    files=[str(CORPUS_PATH)],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    limit_alphabet=1000,
    wordpieces_prefix="##",
    special_tokens=[
        "[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"
    ],
)

# 저장 (tokenizers 포맷)
tokenizer_trainer.save_model(str(TOKENIZER_DIR))
print("토크나이저 학습 및 저장 완료:", TOKENIZER_DIR.resolve())


토크나이저 학습 및 저장 완료: C:\workspace\multi02_data_science\project\simple_kcbert\kcbert_tokenizer


## 4. Hugging Face 토크나이저 래핑

In [5]:
from transformers import BertTokenizerFast

# tokenizers로 만든 vocab.txt를 기반으로 HF 토크나이저 래핑
tokenizer = BertTokenizerFast(
    vocab_file=str(TOKENIZER_DIR / "vocab.txt"),
    do_lower_case=LOWER_CASE,
    do_basic_tokenize=True
)
tokenizer.save_pretrained(TOKENIZER_DIR)
print("HF BertTokenizerFast 저장 완료:", TOKENIZER_DIR.resolve())

# 간단 테스트
print(tokenizer.tokenize("이 문장은 한국어 BERT 토크나이저 테스트입니다."))


c:\miniconda3\envs\kcbert\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HF BertTokenizerFast 저장 완료: C:\workspace\multi02_data_science\project\simple_kcbert\kcbert_tokenizer
['이', '문장', '##은', '한국어', 'B', '##ER', '##T', '토크', '##나이', '##저', '테스트', '##입니다', '.']


## 5. 데이터셋 구성 (MLM용)

In [6]:

from datasets import load_dataset

# 0) 전제: tokenizer는 미리 로드돼 있어야 함
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
# BLOCK_SIZE = 128

# 1) 텍스트 파일 로드 (기본 split 이름: "train")
raw_datasets = load_dataset("text", data_files=str(CORPUS_PATH))

# 2) 빈 줄/None 제거 (윈도우 줄바꿈 포함)
def _is_valid(example):
    t = example["text"]
    return isinstance(t, str) and t.strip() != ""

raw_datasets = raw_datasets.filter(_is_valid)

# 3) 토크나이즈 함수 (변수명 tokenizer로 통일)
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=BLOCK_SIZE,
        return_special_tokens_mask=True,
    )

# 컬럼명 자동 파악 (text 외에 다른 컬럼이 생겼을 경우 대응)
remove_cols = raw_datasets["train"].column_names

# Windows 안전을 위해 num_proc 파라미터 아예 빼버림 (단일 프로세스)
tokenized = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=remove_cols
)

# 4) BLOCK_SIZE로 길이 맞춰 묶기
def group_texts(examples):
    # 모든 리스트를 이어 붙인 뒤 BLOCK_SIZE 단위로 자름
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [t[i:i+BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    return result

lm_datasets = tokenized.map(
    group_texts,
    batched=True
)

train_dataset = lm_datasets["train"]
print(train_dataset)

Generating train split: 2790471 examples [00:06, 400513.01 examples/s]
Map: 100%|██████████| 2790471/2790471 [11:17<00:00, 4120.17 examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 555996
})


## 6. Data Collator (MLM 마스킹)

In [7]:

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROB
)

## 7. 작은 BERT 구성 및 초기화

In [8]:

from transformers import BertConfig, BertForMaskedLM

config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=HIDDEN_SIZE,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=NUM_ATTENTION_HEADS,
    intermediate_size=INTERMEDIATE_SIZE,
    max_position_embeddings=MAX_POS_EMBED,
    type_vocab_size=2,  # NSP에 사용되는 segment id 크기. MLM만 써도 2로 두는게 일반적
    pad_token_id=tokenizer.pad_token_id
)

model = BertForMaskedLM(config)
print("모델 파라미터 수:", sum(p.numel() for p in model.parameters())/1e6, "M")


모델 파라미터 수: 35.858176 M


## 8. 학습 설정 및 실행

In [9]:

from transformers import TrainingArguments, Trainer
import math

args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    overwrite_output_dir=True,
    do_train=True,
    do_eval=False,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    fp16=True,                 # 지원 GPU에서 자동 혼합정밀도 활성화
    gradient_checkpointing=True,  # 메모리 절약
    seed=SEED,
    dataloader_num_workers=2,
    report_to=[],              # wandb 등 로거 끔
)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

train_result = trainer.train()
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(MODEL_DIR)

print("학습 완료. 모델/토크나이저 저장:", MODEL_DIR.resolve())


Step,Training Loss
250,10.137400
500,9.545300
750,9.090500
1000,8.799000
1250,8.677700
1500,8.577200
1750,8.492600
2000,8.410600
2250,8.339300
2500,8.277100


학습 완료. 모델/토크나이저 저장: C:\workspace\multi02_data_science\project\simple_kcbert\kcbert_mlm


## 9. 간이 테스트 (fill-mask)

In [10]:

from transformers import pipeline

mask_filler = pipeline("fill-mask", model=str(MODEL_DIR), tokenizer=str(MODEL_DIR), device_map="auto")
test_sentence = "배송도 빠르고 [MASK]대비 상품도 괜찮았습니다"
for pred in mask_filler(test_sentence):
    print(f"{pred['sequence']}  |  score={pred['score']:.4f}")


Device set to use cuda:0


배송도 빠르고 가격 대비 상품도 괜찮았습니다  |  score=0.0741
배송도 빠르고, 대비 상품도 괜찮았습니다  |  score=0.0386
배송도 빠르고 게임 대비 상품도 괜찮았습니다  |  score=0.0309
배송도 빠르고 상품 대비 상품도 괜찮았습니다  |  score=0.0305
배송도 빠르고 제품 대비 상품도 괜찮았습니다  |  score=0.0200


## 10. 추가 사전학습(도메인 적응, DAPT)

In [14]:
# === DAPT-1) 경로/하이퍼 설정 + 베이스 모델 불러오기 ===
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForMaskedLM

# 기존 노트북에서 정의한 MODEL_DIR 사용. 없으면 기본값으로 대체.
try:
    MODEL_DIR
except NameError:
    MODEL_DIR = Path("./kcbert_mlm")  # 필요시 수정

DAPT_OUT_DIR = Path(MODEL_DIR) / "dapt"
DAPT_CORPUS_PATH = Path("DAPT_dataset.txt")  # 줄당 1문장 텍스트 파일

# 하이퍼파라미터 (DAPT는 약하고 짧게)
BLOCK_SIZE    = 128
MLM_PROB      = 0.15
BATCH_SIZE    = 16
EPOCHS        = 1       # 1~2 권장
LEARNING_RATE = 5e-5
WARMUP_RATIO  = 0.06
WEIGHT_DECAY  = 0.01
LR_SCHED      = "linear"
MAX_GRAD_NORM = 1.0

LOGGING_STEPS = 200
SAVE_STEPS    = 1000
SAVE_LIMIT    = 2
SEED          = 7

USE_FP16      = True
GRAD_CKPT     = True
DATALOADER_WORKERS = 2

# 기존 사전학습 결과 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model     = AutoModelForMaskedLM.from_pretrained(MODEL_DIR)
tokenizer.model_max_length = BLOCK_SIZE  # 경고 방지

print("Loaded base model for DAPT from:", Path(MODEL_DIR).resolve())
print("DAPT corpus path:", Path(DAPT_CORPUS_PATH).resolve())

Loaded base model for DAPT from: C:\workspace\multi02_data_science\project\simple_kcbert\kcbert_mlm
DAPT corpus path: C:\workspace\multi02_data_science\project\simple_kcbert\DAPT_dataset.txt


In [15]:
# === DAPT-2) 데이터 로드/토크나이즈/고정길이 묶기 ===
from datasets import load_dataset
from itertools import chain

# 1) 라인 단위 텍스트 로드
raw = load_dataset("text", data_files=str(DAPT_CORPUS_PATH), split="train")

# 2) 공백/빈 줄 제거
def _valid(ex):
    t = ex["text"]
    return isinstance(t, str) and t.strip() != ""

raw = raw.filter(_valid)

# 3) 토크나이즈
def tok_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=BLOCK_SIZE,
        return_special_tokens_mask=True,
    )

tok = raw.map(tok_fn, batched=True, remove_columns=["text"])

# 4) 토큰을 이어붙여 BLOCK_SIZE로 분할
def group_texts(examples):
    concatenated = {k: list(chain.from_iterable(examples[k])) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    return {
        k: [t[i:i+BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }

train_dataset = tok.map(group_texts, batched=True)
print(train_dataset)


Generating train split: 199725 examples [00:00, 232540.04 examples/s]
Map: 100%|██████████| 199725/199725 [00:28<00:00, 7034.88 examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 66781
})


In [16]:
# === DAPT-3) Collator/Trainer 설정 ===
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=MLM_PROB
)

args = TrainingArguments(
    output_dir=str(DAPT_OUT_DIR),
    overwrite_output_dir=True,
    do_train=True,
    do_eval=False,                       # 필요시 dev 구성해 True로 변경
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_LIMIT,
    fp16=USE_FP16,
    gradient_checkpointing=GRAD_CKPT,
    seed=SEED,
    dataloader_num_workers=DATALOADER_WORKERS,
    report_to=[],
    save_safetensors=True,
    lr_scheduler_type=LR_SCHED,
    max_grad_norm=MAX_GRAD_NORM,
)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21308\2593477346.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [17]:
# === DAPT-4) 학습 실행 + 저장 ===
train_result = trainer.train()
trainer.save_model(str(DAPT_OUT_DIR))
tokenizer.save_pretrained(str(DAPT_OUT_DIR))
print("DAPT complete. Saved to:", Path(DAPT_OUT_DIR).resolve())

Step,Training Loss
200,4.712700
400,4.308100
600,4.098800
800,4.025900
1000,3.911400
1200,3.838400
1400,3.808600
1600,3.758100
1800,3.726100
2000,3.686000


DAPT complete. Saved to: C:\workspace\multi02_data_science\project\simple_kcbert\kcbert_mlm\dapt


In [19]:
# === DAPT-5) 간단 품질 점검 (선택) ===
mask_filler = pipeline("fill-mask", model=str(DAPT_OUT_DIR), tokenizer=str(DAPT_OUT_DIR), device_map="auto")
test_sentence = "배송도 빠르고 [MASK]대비 상품도 괜찮았습니다"
for pred in mask_filler(test_sentence):
    print(f"{pred['sequence']}  |  score={pred['score']:.4f}")

Device set to use cuda:0


배송도 빠르고 가격 대비 상품도 괜찮았습니다  |  score=0.8653
배송도 빠르고 제품 대비 상품도 괜찮았습니다  |  score=0.0362
배송도 빠르고 상품 대비 상품도 괜찮았습니다  |  score=0.0112
배송도 빠르고 성능 대비 상품도 괜찮았습니다  |  score=0.0096
배송도 빠르고 배송 대비 상품도 괜찮았습니다  |  score=0.0081


In [23]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("Fine_tuning_shopping.txt")

# 탭으로 구분, 헤더 없음
df = pd.read_csv(DATA_PATH, sep="\t", header=None, names=["text", "label"], dtype={"text": str})
# 공백/결측 제거
df = df.dropna(subset=["text"]).reset_index(drop=True)
df["text"] = df["text"].str.strip()
# 라벨을 int로 강제
df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)

print(df.head(3))
print(df["label"].value_counts())


                                                text  label
0  시계가 약이 없는지 안가네요. 검수후 보내주셨음 좋았을거 같아요. 선물인데 시간이 ...      0
1                                   재질도 별로도 차에 맞지않네요      0
2                    완전 투명한줄 알았는데 노란빛깔이 띄어요 ㅠ너무 아쉬워요      1
label
1    117454
0     72751
Name: count, dtype: int64


In [24]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    df, test_size=0.1, random_state=7, stratify=df["label"]
)
len(train_df), len(valid_df)


(171184, 19021)

In [42]:
from transformers import AutoTokenizer
from pathlib import Path

DAPT_DIR = Path("./kcbert_mlm/dapt")  # DAPT 저장 폴더
tokenizer = AutoTokenizer.from_pretrained(DAPT_DIR, use_fast=True)
tokenizer.model_max_length = 128

print(tokenizer.name_or_path)

kcbert_mlm\dapt


In [65]:
from transformers import AutoModelForSequenceClassification

num_labels = 2
model_cls = AutoModelForSequenceClassification.from_pretrained(
    DAPT_DIR,
    num_labels=num_labels,
    problem_type="single_label_classification",
    id2label={0:"NEG", 1:"POS"},
    label2id={"NEG":0, "POS":1},
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at kcbert_mlm\dapt and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [66]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df, preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,   # 파인튜닝에선 128이 보통 충분
    )

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
valid_tok = valid_ds.map(tokenize_fn, batched=True, remove_columns=["text"])


Map: 100%|██████████| 19021/19021 [00:01<00:00, 17558.19 examples/s]


In [67]:
print(model_cls.name_or_path)

kcbert_mlm\dapt


In [68]:
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
import numpy as np

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": p, "recall": r}


In [69]:
from transformers import TrainingArguments, Trainer

FT_BATCH_SIZE   = 32
FT_EPOCHS       = 3
FT_LR           = 3e-5

args = TrainingArguments(
    output_dir=str(MODEL_DIR / "finetune"),
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,                         # 학습 후 evaluate() 호출
    per_device_train_batch_size=FT_BATCH_SIZE,
    per_device_eval_batch_size=FT_BATCH_SIZE,
    learning_rate=FT_LR,
    weight_decay=0.01,
    num_train_epochs=FT_EPOCHS,
    warmup_ratio=0.06,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    seed=7,
    dataloader_num_workers=2,
    report_to=[],
)

trainer = Trainer(
    model=model_cls,
    args=args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21308\516827587.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [71]:
train_out = trainer.train()
eval_out  = trainer.evaluate()
print(eval_out)

# 최종 저장
trainer.save_model(str(MODEL_DIR / "finetune"))
tokenizer.save_pretrained(str(MODEL_DIR / "finetune"))


Step,Training Loss
100,0.679500
200,0.609700
300,0.454800
400,0.413200
500,0.403700
600,0.374100
700,0.365800
800,0.361700
900,0.323000
1000,0.334700


{'eval_loss': 0.2618907392024994, 'eval_accuracy': 0.8953262183902003, 'eval_f1': 0.9153018249883014, 'eval_precision': 0.9147181362129071, 'eval_recall': 0.9158862591520518, 'eval_runtime': 34.9943, 'eval_samples_per_second': 543.546, 'eval_steps_per_second': 17.003, 'epoch': 3.0}


('kcbert_mlm\\finetune\\tokenizer_config.json',
 'kcbert_mlm\\finetune\\special_tokens_map.json',
 'kcbert_mlm\\finetune\\vocab.txt',
 'kcbert_mlm\\finetune\\added_tokens.json',
 'kcbert_mlm\\finetune\\tokenizer.json')

In [73]:
from transformers import TextClassificationPipeline

pipe = TextClassificationPipeline(
    model=model_cls, tokenizer=tokenizer, device=0 if args.fp16 else -1, truncation=True
)

samples = [
    "배송 진짜 빨랐고 포장 깔끔. 다음에도 여기서 산다.",
    "색깔이 화면이랑 다르고 질도 별로였음. 비추.",
    "시계가 약이 없는지 안 가네요…",
    "그렇게 별로인 제품은 아닌거같아요"
]
for s in samples:
    print(s, "->", pipe(s))


Device set to use cuda:0


배송 진짜 빨랐고 포장 깔끔. 다음에도 여기서 산다. -> [{'label': 'POS', 'score': 0.9973374009132385}]
색깔이 화면이랑 다르고 질도 별로였음. 비추. -> [{'label': 'NEG', 'score': 0.9989392161369324}]
시계가 약이 없는지 안 가네요… -> [{'label': 'POS', 'score': 0.5416815876960754}]
그렇게 별로인 제품은 아닌거같아요 -> [{'label': 'NEG', 'score': 0.9980657696723938}]


In [74]:
eval_out = trainer.evaluate()
print(eval_out)

{'eval_loss': 0.2618907392024994, 'eval_accuracy': 0.8953262183902003, 'eval_f1': 0.9153018249883014, 'eval_precision': 0.9147181362129071, 'eval_recall': 0.9158862591520518, 'eval_runtime': 36.6253, 'eval_samples_per_second': 519.341, 'eval_steps_per_second': 16.246, 'epoch': 3.0}


In [75]:
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

preds = np.argmax(trainer.predict(valid_tok).predictions, axis=-1)
labels = np.array(valid_tok["label"])
cm = confusion_matrix(labels, preds, labels=[0,1])
print(cm)  # [[TN FP],[FN TP]]
print(classification_report(labels, preds, digits=4))

[[ 6272  1003]
 [  988 10758]]
              precision    recall  f1-score   support

           0     0.8639    0.8621    0.8630      7275
           1     0.9147    0.9159    0.9153     11746

    accuracy                         0.8953     19021
   macro avg     0.8893    0.8890    0.8892     19021
weighted avg     0.8953    0.8953    0.8953     19021



In [52]:

# 체크포인트에서 이어서 학습하려면, 위의 Trainer 설정을 재사용하고 아래처럼 경로만 지정합니다.
# 예: 마지막 체크포인트가 'kcbert_mlm/checkpoint-10000' 라면
# resume_from_checkpoint = "kcbert_mlm/checkpoint-10000"
# trainer.train(resume_from_checkpoint=resume_from_checkpoint)
#
# 또는 새로운 도메인 코퍼스 경로를 CORPUS_PATH로 바꾸고, tokenized/lm_datasets만 다시 만들고
# 같은 모델 디렉토리(MODEL_DIR)에서 이어서 train()을 호출하면 됩니다.
pass



## 11. 팁 & 트러블슈팅

- **메모리 부족**: `BATCH_SIZE`를 낮추고, `gradient_accumulation_steps`를 사용하세요. `gradient_checkpointing=True` 유지.
- **속도 향상**: `BLOCK_SIZE`를 512로 늘리면 성능은 좋아질 수 있으나, 메모리 비용이 큽니다. 데이터가 충분할 때만 시도.
- **토크나이저 재학습**: 말뭉치가 바뀌면 `TOKENIZER_DIR`을 비우고 다시 학습하세요.
- **평가**: MLM 퍼플렉서티를 보려면 eval split을 만들어 `Trainer`에 `eval_dataset`과 `compute_metrics`를 달아주세요.
- **한글 전처리**: 불필요한 특수문자 제거, 중복 줄 제거, 문장 단위 분리 등은 코퍼스 품질에 따라 추가하세요.
